In [ ]:
!uv add pillow

Resolved 40 packages in 2.02s
Prepared 1 package in 5.74s
Installed 1 package in 143ms
 + pillow==12.2.0


# Dataset Processing
1. Map the dhanmondi data with physician table for more information about the doctors.
these columns are taken for next process
```python
['PRSID', 'PHYID', 'PHYNM', 'PHYDEGR', 'PHY_SPC', 'PHY_DES', 'INS_NM', 'INS_ADD', 'INS_THA', 'INS_DST', 'CH_ADD', 'CH_DST', 'CH_THA', 'PHY_GEND', 'BMDC_REGNO', 'PHYNM_DT', 'CHNM_DT', 'IMAGE_PATH']
```
2. Pre-process images for fintuning (gray scale convertion, height,width ratio solve, contrast increase)

3. Prepare the image `ground truth` using mapped data.

## 1. Dataset mapping with physician table

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

output_path = "data/processed"

In [20]:
import pandas as pd
import os
import re

def extract_pr(path):
    if not isinstance(path, str):
        return None
    # Extract the filename part starting with PR (case insensitive just in case, but usually uppercase)
    match = re.search(r'(PR[A-Z0-9_]+)', os.path.basename(path), re.IGNORECASE)
    if match:
        return match.group(1)
    return None
def process_data(output_path):
    # Paths
    csv_path = "data/doctor_image_details (1).csv"
    excel_path = "data/R232C6_Dhanmondi_Data (2).xlsx"
    output_path = f"{output_path}/mapped_doctor_data.csv"

    print(f"Loading CSV: {csv_path}")
    df_csv = pd.read_csv(csv_path)
    print(f"CSV loaded. Sample data:\n{df_csv.head(2)}")

    # Extract PR from path
    # User says: "starting with pr is the pr"
    # Example: /content/drive/MyDrive/All_Image/PR232C6DHK23_P001.jpg -> PR232C6DHK23_P001
    

    df_csv['extracted_pr'] = df_csv['IMAGE_NAME_WITH_PATH'].apply(extract_pr)
    print(f"Extracted PR sample: {df_csv['extracted_pr'].head().tolist()}")

    print(f"Loading Excel: {excel_path}")
    # Load Excel - using the first sheet by default as checked before
    df_xl = pd.read_excel(excel_path)
    print(f"Excel loaded. Sample data:\n{df_xl.head(2)}")
    
    # Let's check for case insensitive match too
    df_xl['IMG_NM_upper'] = df_xl['IMG_NM'].astype(str).str.upper()
    df_csv['extracted_pr_upper'] = df_csv['extracted_pr'].astype(str).str.upper()

    print("Merging with Dhanmondi Data...")
    # Join on IMG_NM as it's the actual link between the files
    merged_df = pd.merge(
        df_csv, 
        df_xl, 
        left_on='extracted_pr_upper', 
        right_on='IMG_NM_upper', 
        how='left'
    )

    # Now load Physicianlist
    physician_list_path = "data/Physicianlist.xlsx"
    print(f"Loading Physician List: {physician_list_path}")
    df_physician = pd.read_excel(physician_list_path)
    
    # In Dhanmondi data it is 'PHYID', in Physicianlist it is 'PHY_ID'
    print("Merging with Physician List...")
    merged_df = pd.merge(
        merged_df,
        df_physician,
        left_on='PHYID',
        right_on='PHY_ID',
        how='left',
        suffixes=('', '_phy')
    )
    merged_df["IMAGE_PATH"] = merged_df['IMAGE_NAME_WITH_PATH'].str.split('/').str[-1]  # Keep only the filename for clarity
    # Clean up temporary columns and unwanted columns
    columns_to_drop = [
        'IMAGE_NAME_WITH_PATH','extracted_pr', 'extracted_pr_upper', 'IMG_NM_upper',
        'DOCTOR_DETAILS_COMBINED', 'MONTH', 'ROUND', 'YEAR', 'BOOKID', 'SHOPID', 
        'CDATE', 'PDATE', 'PRSTYPE', 'PSCSLNO', 'PHY_ID', 'PHY_NM', 'PHY_DEG',
        'VC2', 'NAME', 'GP', 'QTPRS', 'QTPURCH', 'CYCLE', 'FICODE', 'OPERATOR', 
        'DIAGCD', 'DIAGNAME', 'DIAGOPTR', 'DIAGEDTR', 'GENDER', 'AGE', 'PHYSPCD', 
        'CINSTCD', 'EDATE', 'ETIME', 'DIAEDATE', 'DIAETIME', 'SCHDSLT', 'FSCODE', 
        'EDITOR', 'EDDATE', 'ROUND_phy', 'CINSTCD_phy', 'MCODE', 'MARKET', 
        'PHYSP_C', 'FICODE_phy', 'SC', 'NOTE', 'PD03', 'PD04', 'DUPLICATE', 
        'OLDCODE', 'SHEETNO', 'EDITDATE', 'EDITOR_phy', 'UNICODE', 'CYCLE_phy', 
        'DSDCODE', 'MCHCODE', 'CH_PHNO1', 'CH_PHNO2', 'CH_PHNO3', 'PHY_PHNO', 
        'PHYEMAIL', 'PHYNM_DT_DUP', 'PHYNM_ALL_DUP', 'CHNM_DT_DUP', 'CHNM_ALL_DUP','IMG_NM','DIVISION','HINSTCD','OPERATO','REGION','DT'
    ]
    
    # Filter columns_to_drop to only those that exist in the dataframe
    existing_drops = [c for c in columns_to_drop if c in merged_df.columns]
    merged_df = merged_df.drop(columns=existing_drops)

    print(f"Merge complete. Rows in CSV: {len(df_csv)}, Rows in Merged: {len(merged_df)}")
    print(f"Matched rows in Dhanmondi: {merged_df['PRSID'].notna().sum() if 'PRSID' in merged_df.columns else 'N/A'}")
    print(f"Remaining columns: {merged_df.columns.tolist()}")

    # Save to CSV
    merged_df.to_csv(output_path, index=False)
    print(f"Result saved to: {output_path}")


In [21]:
output_path = "data/processed"
process_data(output_path)

Loading CSV: data/doctor_image_details (1).csv


FileNotFoundError: [Errno 2] No such file or directory: 'data/doctor_image_details (1).csv'

## 2. Image Processing

In [34]:
from PIL import ImageEnhance
from PIL import Image
import pandas as pd
import os

def process_image(image,max_width):
    """
    1. convert to gray scale
    2. resize to max_width while maintaining aspect ratio
    3. increase contrast
    """
    image = image.convert('L')  # Convert to grayscale
    
    if image.width > max_width:
        aspect_ratio = image.height / image.width
        new_height = int(max_width * aspect_ratio)
        image = image.resize((max_width, new_height))  # Resize while maintaining aspect ratio

    # Increase contrast
    image_enhanced = ImageEnhance.Contrast(image)
    image_enhanced = image_enhanced.enhance(2)  # Adjust the contrast level (2 is an example)
    return image_enhanced
    
def preprocess_images(source_dir, output_dir,max_width=512):
    image_paths = [os.path.join(source_dir, f) for f in os.listdir(source_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    image_paths = image_paths  # Process only the first 10 images
    for image_path in image_paths:
        image = Image.open(image_path)
        processed_image = process_image(image, max_width=max_width)
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, os.path.basename(image_path))
        processed_image.save(output_path,format='JPEG', quality=85, optimize=True)        

In [35]:
jsonl_path = "/content/drive/MyDrive/Work_with_sazzad_vai/Dataset/ground_truth.jsonl"
source_directory = '/content/drive/MyDrive/All_Image'
output_directory = '/content/drive/MyDrive/All_Image_Processed'
max_width = 512
preprocess_images(source_directory, output_directory)

## 3. Ground truth prepare

In [33]:
import pandas as pd

In [36]:
df= pd.read_csv(f"/content/drive/MyDrive/Work_with_sazzad_vai/Dataset/mapped_doctor_data.csv")
df.shape


(9640, 24)

In [37]:

print(df.shape)
df.drop_duplicates(inplace=True)
print(df.shape)

(9640, 24)
(1839, 24)


In [38]:
df.head()

,PRSID,PHYID,PHYNM,PHYDEGR,IMG_NM,DIVISION,PHY_SPC,PHY_DES,HINSTCD,INS_NM,...,CH_DST,CH_THA,OPERATO,REGION,DT,PHY_GEND,BMDC_REGNO,PHYNM_DT,CHNM_DT,image_path
0,PRS232C6016929,DHA28408,DR S M SIDDIQUR RAHMAN,"MBBS, D-CARD, MD, FACC",PR232C6DHK23_P001,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PR232C6DHK23_P001.jpg
6,PRS232C6016937,DHK00929,DR MD. NUR HOSSAIN,"MBBS, MD",PR232C6DHK23_P002,1-DHK,CARDIOLOGY,PROFESSOR,H0320,NICVD,...,DHAKA,DHANMONDI,DILIP,DHK,DHAKADHANMONDI,M,NaN,DRMDNURHOSSAIN,MEDINOVA,PR232C6DHK23_P002.jpg
14,PRS232C6017997,DHK96865,DR MD FAIZUL ISLAM CHOWDHURY,"MBBS, FCPS, PHD, WHO",PR232C6DHK23_P003,1-DHK,MEDICINE,PROFESSOR,H0100,DHAKA MEDICAL COLLEGE HOSPITAL,...,DHAKA,DHANMONDI,4PDCD,DHK,DHAKADHANMONDI,M,A13255,DRMDFAIZULISLAMCHOWDHURY,MEDINOVA,PR232C6DHK23_P003.jpg
20,PRS232C6018008,DHK40007,DR MD. FAZLUL KADIR,"MBBS, FCPS",PR232C6DHK23_P004,1-DHK,MEDICINE,PROFESSOR,H0070,BANGLADESH MEDICAL COLLEGE HOSPITAL,...,DHAKA,DHANMONDI,4PBLS,DHK,DHAKADHANMONDI,M,NaN,DRMDFAZLULKADIR,MEDINOVA,PR232C6DHK23_P004.jpg
27,PRS232C6018026,DHK50102,DR M. ABU HENA CHOWDHURY,"MBBS, FCPS, DDV",PR232C6DHK23_P005,1-DHK,"SKIN, VD",ASSOCIATE PROFESSOR,H0110,"BSMMU, SHAHBAG",...,DHAKA,DHANMONDI,4PMRZ,DHK,DHAKADHANMONDI,M,NaN,DRMABUHENACHOWDHURY,IBN SINA MEDICAL IMAGING CENTER,PR232C6DHK23_P005.jpg


In [39]:
# # check how many rows have  PRSID, PHYID, PHYNM and PHYDEGR only
## drop rows where all of these columns are null
df_pure = df.dropna(subset=['PHY_SPC', 'PHY_DES', 'INS_NM', 'INS_ADD', 'INS_THA'], how='all')
df_pure.shape
# df.dropna(df[df[['PHY_SPC', 'PHY_DES', 'INS_NM', 'INS_ADD', 'INS_THA']].isnull().all(axis=1)])

(1642, 24)

#### Finetune data format

In [43]:
import json

# Output path for JSONL
jsonl_path = "/content/drive/MyDrive/Work_with_sazzad_vai/Dataset/ground_truth.jsonl"
exclude_fields = ["IMAGE_PATH","image_path", "PRSID", "PHYID"]
i = 0 
with open(jsonl_path, "w", encoding="utf-8") as f:
    for idx, row in df_pure.iterrows():

        # Convert row to dict and keep empty fields as empty string
        row_dict = {col: ("" if pd.isna(val) else val) for col, val in row.items()}
        # print(f"Processing row {idx}: {row_dict}")  # Debug print to check the content of each row
        row_dict
        i += 1
        record = {
            "idx": int(i),
            "image_path": row_dict.get("image_path", ""),
            "output": {k: v for k, v in row_dict.items() if k not in exclude_fields}
        }
        if i < 5:  # Print the first few records for verification
            print(f"Constructed record for row {i}: {record}")  # Debug print to check the constructed record

        f.write(json.dumps(record, ensure_ascii=False,default=str) + "\n")

print(f"JSONL file saved: {jsonl_path}")

Constructed record for row 1: {'idx': 1, 'image_path': 'PR232C6DHK23_P002.jpg', 'output': {'PHYNM': 'DR MD. NUR HOSSAIN', 'PHYDEGR': 'MBBS, MD', 'IMG_NM': 'PR232C6DHK23_P002', 'DIVISION': '1-DHK', 'PHY_SPC': 'CARDIOLOGY', 'PHY_DES': 'PROFESSOR', 'HINSTCD': 'H0320', 'INS_NM': 'NICVD', 'INS_ADD': 'NICVD HOSPITAL', 'INS_THA': 'MOHAMMADPUR', 'INS_DST': 'DHAKA', 'CH_ADD': 'MEDINOVA, H-71/A, R-5/A, DHANMONDI R/A', 'CH_DST': 'DHAKA', 'CH_THA': 'DHANMONDI', 'OPERATO': 'DILIP', 'REGION': 'DHK', 'DT': 'DHAKADHANMONDI', 'PHY_GEND': 'M', 'BMDC_REGNO': '', 'PHYNM_DT': 'DRMDNURHOSSAIN', 'CHNM_DT': 'MEDINOVA'}}
Constructed record for row 2: {'idx': 2, 'image_path': 'PR232C6DHK23_P003.jpg', 'output': {'PHYNM': 'DR MD FAIZUL ISLAM CHOWDHURY', 'PHYDEGR': 'MBBS, FCPS, PHD, WHO', 'IMG_NM': 'PR232C6DHK23_P003', 'DIVISION': '1-DHK', 'PHY_SPC': 'MEDICINE', 'PHY_DES': 'PROFESSOR', 'HINSTCD': 'H0100', 'INS_NM': 'DHAKA MEDICAL COLLEGE HOSPITAL', 'INS_ADD': 'SECRETARIET ROAD, RAMNA', 'INS_THA': 'RAMNA', 'INS_DST

### task : finetune data format


In [44]:
Task_prompt1 = """
You are a professional Medical Prescription Information Extractor.

Extract the following information from the prescription image:

- PHYNM (Doctor Name)
- PHYDEGR (Doctor Degrees)
- PHY_SPC (Specialty)
- PHY_DES (Designation)
- BMDC_REGNO (BMDC Registration Number)
- INS_NM (Hospital / Institution Name)
- INS_ADD (Hospital Address)
- INS_THA (Hospital Thana)
- INS_DST (Hospital District)
- CH_ADD (Chamber Address)
- CH_THA (Chamber Thana)
- CH_DST (Chamber District)

Rules:
1. Extract text exactly as written.
2. Preserve capitalization.
3. Return empty string if a field is missing.
4. Output only valid JSON.
5. Do not generate explanations.
""".strip()

Task_prompt2 = """
You are a professional Medical Prescription OCR Extractor.

Extract doctor, institution, and chamber information from the prescription image.

Use both:
- visible text
- document layout and spatial relationships

Extract only information that appears in the image.

Return the result as valid JSON using this schema:

{
    "PHYNM": "",
    "PHYDEGR": "",
    "PHY_SPC": "",
    "PHY_DES": "",
    "BMDC_REGNO": "",
    "INS_NM": "",
    "INS_ADD": "",
    "INS_THA": "",
    "INS_DST": "",
    "CH_ADD": "",
    "CH_THA": "",
    "CH_DST": ""
}

Rules:
- Preserve original spelling and capitalization.
- Do not normalize or correct text.
- Do not infer missing information.
- Use empty string ("") for unavailable fields.
- Return JSON only.
""".strip()



In [ ]:
llm_finetune_data = []
train_ds= []
val_ds = []

image_path_set = set()

for line in open(jsonl_path, "r", encoding="utf-8"):
    record = json.loads(line)
    image_path = record["image_path"]
    output = record["output"]

    if image_path in image_path_set:
        print(f"Duplicate image path found: {image_path}")
    else:
        image_path_set.add(image_path)

    task_1_sft_record = {
        "conversation": [
            {
                "value": "<image>"+Task_prompt1,
                "from": "human"
            },
            {
                "value": record["output"],
                "from": "gpt"
            }
                ],
        "images": [image_path]
        }
    task_2_sft_record = {
        "conversation": [
            {
                "value": "<image>"+Task_prompt2,
                "from": "human"
            },
            {         "value": record["output"],    "from": "gpt"            }
                ],  
        "images": [image_path]
    }
    